In [11]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import json

In [21]:
# --- 1) Load dataset ---
print("Upload CSV now (use the file chooser) OR set path to your CSV in csv_path variable.")
# Option A: upload manually
from google.colab import files
uploaded = files.upload()  # use file picker and upload ai_status_forecast_dataset_4600.csv
# After upload, get filename:
csv_path = list(uploaded.keys())[0] if uploaded else "/content/ai_status_forecast_dataset_4600.csv"
print("Loaded:", csv_path)

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
print(df.columns.tolist())
print(df['project_status'].value_counts())

Upload CSV now (use the file chooser) OR set path to your CSV in csv_path variable.


Saving ai_status_forecast_dataset_4600.csv to ai_status_forecast_dataset_4600 (2).csv
Loaded: ai_status_forecast_dataset_4600 (2).csv
Dataset shape: (4600, 14)
['milestones_total', 'milestones_completed', 'milestones_completion_pct', 'milestones_on_time_pct', 'time_elapsed_pct', 'schedule_variance', 'delivery_velocity', 'budget_used_pct', 'budget_efficiency', 'resource_availability', 'resource_availability_code', 'collaborators', 'sdg_confidence_avg', 'project_status']
project_status
At Risk      2280
On Track     2149
Excellent     132
Delayed        39
Name: count, dtype: int64


In [33]:
# --- 2) Define training features that match frontend inputs ---
# Frontend sends: milestones_pct (0..1), time_elapsed_pct (0..1),
# collaborators (int), resource_availability (Low/Medium/High), budget_pct (0..100)
# We'll also accept optional sdg_confidence (0..1). If dataset has sdg_confidence_avg use that.

# Map resource availability textual to numeric codes:
ra_map = {"Low":0, "Medium":1, "High":2}
if 'resource_availability' in df.columns:
    df['resource_availability_code'] = df['resource_availability'].map(ra_map).fillna(1).astype(int)
# If dataset already has resource_availability_code column, keep it.

# Ensure we have a numeric sdg_confidence_avg; if missing, add default
if 'sdg_confidence_avg' not in df.columns:
    df['sdg_confidence_avg'] = 0.85

# We'll use these base inputs (some already exist in dataset)
BASE_FEATURES = [
    'milestones_completion_pct',   # -> frontend milestones_pct
    'time_elapsed_pct',
    'collaborators',
    'resource_availability_code',
    'budget_used_pct',            # -> frontend budget_pct
    'sdg_confidence_avg'
]

In [34]:
# Derived features to improve model (computed from inputs)
def add_derived_features(df):
    df = df.copy()
    # schedule variance: completion - elapsed
    df['schedule_variance'] = df['milestones_completion_pct'] - df['time_elapsed_pct']
    # budget_efficiency: completion / budget (avoid divide by zero)
    df['budget_efficiency'] = df['milestones_completion_pct'] / (df['budget_used_pct'] / 100.0 + 1e-6)
    # delivery_velocity: completion_pct / time_elapsed_pct (scaled)
    df['delivery_velocity'] = df['milestones_completion_pct'] / (df['time_elapsed_pct'].clip(lower=1e-6))
    return df

df = add_derived_features(df)

In [35]:
# Final feature list used for training
FEATURES = [
    'milestones_completion_pct','time_elapsed_pct','collaborators',
    'resource_availability_code','budget_used_pct','sdg_confidence_avg',
    'schedule_variance','budget_efficiency','delivery_velocity'
]

In [36]:
# Drop rows with NaN in features (if any)
df = df.dropna(subset=FEATURES + ['project_status'])
print("After dropna:", df.shape)

After dropna: (4600, 14)


In [37]:
# --- 3) Encode target labels ---
le = LabelEncoder()
y = le.fit_transform(df['project_status'])   # maps labels to ints
print("Label classes:", le.classes_)

X = df[FEATURES].astype(float).values

Label classes: ['At Risk' 'Delayed' 'Excellent' 'On Track']


In [38]:
# --- 4) Train/Test split (stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

In [39]:
# --- 5) Build pipeline: scaler + XGBoost classifier ---
scaler = StandardScaler()
xgb = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    use_label_encoder=False,
    num_class=len(le.classes_),
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs= -1
)

pipeline = Pipeline([
    ('scaler', scaler),
    ('xgb', xgb)
])

In [41]:
# --- 6) Train with early stopping on a small validation split from train ---
# We'll pass eval_set for early stopping through the underlying estimator:
# create a validation split from X_train
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=24)

# Fit the XGB with eval_set argument
pipeline.named_steps['xgb'].fit(
    scaler.fit_transform(X_tr),
    y_tr,
    eval_set=[(scaler.transform(X_val), y_val)]
)

# Because we trained the estimator directly above, we must set the pipeline steps accordingly.
# (scaler already fit; xgb fitted). To make full pipeline usable, we'll re-create pipeline with fitted components:
pipeline = Pipeline([('scaler', scaler), ('xgb', pipeline.named_steps['xgb'])])

[0]	validation_0-mlogloss:1.12460
[1]	validation_0-mlogloss:1.05813
[2]	validation_0-mlogloss:0.99691
[3]	validation_0-mlogloss:0.94093
[4]	validation_0-mlogloss:0.89108
[5]	validation_0-mlogloss:0.84519
[6]	validation_0-mlogloss:0.80127
[7]	validation_0-mlogloss:0.76031
[8]	validation_0-mlogloss:0.72073
[9]	validation_0-mlogloss:0.68432
[10]	validation_0-mlogloss:0.64981
[11]	validation_0-mlogloss:0.61949
[12]	validation_0-mlogloss:0.59092
[13]	validation_0-mlogloss:0.56255
[14]	validation_0-mlogloss:0.53544
[15]	validation_0-mlogloss:0.51086
[16]	validation_0-mlogloss:0.48748
[17]	validation_0-mlogloss:0.46533
[18]	validation_0-mlogloss:0.44424
[19]	validation_0-mlogloss:0.42459
[20]	validation_0-mlogloss:0.40690


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:27:16] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[21]	validation_0-mlogloss:0.38935
[22]	validation_0-mlogloss:0.37275
[23]	validation_0-mlogloss:0.35703
[24]	validation_0-mlogloss:0.34226
[25]	validation_0-mlogloss:0.32910
[26]	validation_0-mlogloss:0.31602
[27]	validation_0-mlogloss:0.30324
[28]	validation_0-mlogloss:0.29175
[29]	validation_0-mlogloss:0.28036
[30]	validation_0-mlogloss:0.26964
[31]	validation_0-mlogloss:0.25995
[32]	validation_0-mlogloss:0.25030
[33]	validation_0-mlogloss:0.24157
[34]	validation_0-mlogloss:0.23453
[35]	validation_0-mlogloss:0.22635
[36]	validation_0-mlogloss:0.21872
[37]	validation_0-mlogloss:0.21134
[38]	validation_0-mlogloss:0.20448
[39]	validation_0-mlogloss:0.19719
[40]	validation_0-mlogloss:0.19052
[41]	validation_0-mlogloss:0.18498
[42]	validation_0-mlogloss:0.17954
[43]	validation_0-mlogloss:0.17376
[44]	validation_0-mlogloss:0.16868
[45]	validation_0-mlogloss:0.16408
[46]	validation_0-mlogloss:0.15976
[47]	validation_0-mlogloss:0.15500
[48]	validation_0-mlogloss:0.15076
[49]	validation_0-ml

In [42]:
# --- 7) Evaluate model ---
y_pred = pipeline.predict(X_test)
print("\nClassification report (test):")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)



Classification report (test):
              precision    recall  f1-score   support

     At Risk       1.00      1.00      1.00       456
     Delayed       0.89      1.00      0.94         8
   Excellent       0.67      0.46      0.55        26
    On Track       0.97      0.98      0.98       430

    accuracy                           0.98       920
   macro avg       0.88      0.86      0.87       920
weighted avg       0.97      0.98      0.97       920

Confusion matrix:
 [[455   1   0   0]
 [  0   8   0   0]
 [  0   0  12  14]
 [  1   0   6 423]]


In [43]:
def predict_example(payload):
    """
    payload keys expected:
      - milestones_pct (0..1)
      - time_elapsed_pct (0..1)
      - collaborators (int)
      - resource_availability (str) OR resource_availability_code (int)
      - budget_pct (0..100)
      - sdg_confidence (0..1) optional
    """
    p = payload.copy()
    # map
    if 'resource_availability' in p:
        p['resource_availability_code'] = ra_map.get(p['resource_availability'], 1)
    p.setdefault('sdg_confidence', 0.85)
    # Build dataframe of features
    df_in = pd.DataFrame([{
        'milestones_completion_pct': float(p.get('milestones_pct', 0.0)),
        'time_elapsed_pct': float(p.get('time_elapsed_pct', 0.0)),
        'collaborators': int(p.get('collaborators', 1)),
        'resource_availability_code': int(p.get('resource_availability_code', 1)),
        'budget_used_pct': float(p.get('budget_pct', p.get('budget_used_pct', 0.0))),
        'sdg_confidence_avg': float(p.get('sdg_confidence', 0.85))
    }])
    df_in = add_derived_features(df_in)
    X_in = df_in[FEATURES].values
    probs = pipeline.predict_proba(X_in)[0]
    pred_idx = int(np.argmax(probs))
    return {"status": le.inverse_transform([pred_idx])[0], "confidence": float(probs[pred_idx])}

In [44]:
print("Sample prediction:", predict_example({
    "milestones_pct": 0.5,
    "time_elapsed_pct": 0.4,
    "collaborators": 3,
    "resource_availability": "Medium",
    "budget_pct": 30,
    "sdg_confidence": 0.85
}))

Sample prediction: {'status': 'On Track', 'confidence': 0.9998254179954529}


In [46]:
# --- 8) Save the trained model and label encoder ---
model_filename = 'project_status_predictor_pipeline.pkl'
label_encoder_filename = 'label_encoder.pkl'

joblib.dump(pipeline, model_filename)
joblib.dump(le, label_encoder_filename)

print(f"Model saved to: {model_filename}")
print(f"Label encoder saved to: {label_encoder_filename}")

Model saved to: project_status_predictor_pipeline.pkl
Label encoder saved to: label_encoder.pkl


In [47]:
# Save model + encoder
model_path = "/content/ai_status_pipeline.joblib"
joblib.dump({
    "pipeline": pipeline,
    "label_encoder": le,
    "features": FEATURES,
    "ra_map": ra_map
}, model_path)
print("Saved pipeline to:", model_path)

Saved pipeline to: /content/ai_status_pipeline.joblib
